<a href="https://colab.research.google.com/github/kcclayton/CSC-417-Project/blob/main/KCClayton_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Learning Final Project | CSC 417

## CSAILVision semantic segmention models

This is a semantic segmentation notebook using an [ADE20K](http://groups.csail.mit.edu/vision/datasets/ADE20K/) pretrained model from the open source project [CSAILVision/semantic-segmentation-pytorch](https://github.com/CSAILVision/semantic-segmentation-pytorch).

For other deep-learning Colab notebooks, visit [tugstugi/dl-colab-notebooks](https://github.com/tugstugi/dl-colab-notebooks).

## Set up environment and mount Drive

This is environment setup. The CSAILVision semantic-segmentation-pytorch repository is cloned (but skips the download if already present) and adds it to Python's import path so `mit_semseg` can be used. It then mounts GoogleDrive and creates project folders for model weights, test images, and outputs, so downloaded files persist across Colab sessions.

In [15]:
import os
from os.path import exists, join, basename, splitext

git_repo_url = 'https://github.com/CSAILVision/semantic-segmentation-pytorch.git'
project_name = splitext(basename(git_repo_url))[0]
if not exists(project_name):
  # clone and install dependencies
  !git clone -q $git_repo_url
  #!cd $project_name && pip install -q -r requirement.txt

import sys
sys.path.append(project_name)
import time
import matplotlib
import matplotlib.pylab as plt
plt.rcParams["axes.grid"] = False

In [16]:
# mount drive to colab session
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/semseg-project'
WEIGHTS_DIR = os.path.join(DATA_DIR, 'weights')
IMAGES_DIR = os.path.join(DATA_DIR, 'images')
OUTPUT_DIR = os.path.join(DATA_DIR, 'output')
for d in (WEIGHTS_DIR, IMAGES_DIR, OUTPUT_DIR):
    os.makedirs(d, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## First model: UperNet101


### Download our first pretrained model: UperNet101

According to [https://github.com/CSAILVision/semantic-segmentation-pytorch#performance](https://github.com/CSAILVision/semantic-segmentation-pytorch#performance),  **UperNet101** was the best performing model. We will use it as our first pretrained model:

In [17]:
ENCODER_NAME_1 = 'resnet101'
DECODER_NAME_1 = 'upernet'
PRETRAINED_ENCODER_MODEL_URL_1 = 'http://sceneparsing.csail.mit.edu/model/pytorch/ade20k-%s-%s/encoder_epoch_50.pth' % (ENCODER_NAME_1, DECODER_NAME_1)
PRETRAINED_DECODER_MODEL_URL_1 = 'http://sceneparsing.csail.mit.edu/model/pytorch/ade20k-%s-%s/decoder_epoch_50.pth' % (ENCODER_NAME_1, DECODER_NAME_1)

pretrained_encoder_file_1 = basename(PRETRAINED_ENCODER_MODEL_URL_1)
if not exists(pretrained_encoder_file_1):
  !wget -q $PRETRAINED_ENCODER_MODEL_URL
pretrained_decoder_file_1 = basename(PRETRAINED_DECODER_MODEL_URL_1)
if not exists(pretrained_decoder_file_1):
  !wget -q $PRETRAINED_DECODER_MODEL_URL

### Prepare UperNet101

Load first pretrained model:

In [18]:
from types import SimpleNamespace
import torch
from mit_semseg.models import ModelBuilder, SegmentationModule
from mit_semseg.dataset import TestDataset
from mit_semseg.utils import colorEncode
from scipy.io import loadmat

# options
options = SimpleNamespace(fc_dim=2048,
                          num_class=150,
                          imgSizes = [300, 400, 500, 600],
                          imgMaxSize=1000,
                          padding_constant=8,
                          segm_downsampling_rate=8)

# create model
builder = ModelBuilder()
net_encoder = builder.build_encoder(arch=ENCODER_NAME_1, weights=pretrained_encoder_file_1,
                                    fc_dim=options.fc_dim)
net_decoder = builder.build_decoder(arch=DECODER_NAME_1, weights=pretrained_decoder_file_1,
                                    fc_dim=options.fc_dim, num_class=options.num_class, use_softmax=True)
segmentation_module = SegmentationModule(net_encoder, net_decoder, torch.nn.NLLLoss(ignore_index=-1))
segmentation_module = segmentation_module.eval()
torch.set_grad_enabled(False)

if torch.cuda.is_available():
  segmentation_module = segmentation_module.cuda()

# test on a given image (defaults to first model and options)
def test(test_image_name, model=None, opts=None):
  if model is None: model = segmentation_module
  if opts is None: opts = options
  dataset_test = TestDataset([{'fpath_img': test_image_name}], opts, max_sample=-1)

  batch_data = dataset_test[0]
  segSize = (batch_data['img_ori'].shape[0], batch_data['img_ori'].shape[1])
  img_resized_list = batch_data['img_data']

  scores = torch.zeros(1, opts.num_class, segSize[0], segSize[1])
  if torch.cuda.is_available():
    scores = scores.cuda()

  for img in img_resized_list:
    feed_dict = batch_data.copy()
    feed_dict['img_data'] = img
    del feed_dict['img_ori']
    del feed_dict['info']
    if torch.cuda.is_available():
      feed_dict = {k: o.cuda() for k, o in feed_dict.items()}

    # forward pass
    pred_tmp = model(feed_dict, segSize=segSize)
    scores = scores + pred_tmp / len(opts.imgSizes)

  _, pred = torch.max(scores, dim=1)
  return pred.squeeze(0).cpu().numpy()

Loading weights for net_encoder


RuntimeError: unexpected EOF, expected 1603233 more bytes. The file might be corrupted.

### Evaluate UperNet101 on a test image

First, download a test image from the internet:

In [ ]:
IMAGE_URL = 'https://raw.githubusercontent.com/tugstugi/dl-colab-notebooks/master/resources/lidl.jpg'

image_file = basename(IMAGE_URL)
!wget -q -O $image_file $IMAGE_URL

plt.figure(figsize=(10, 5))
plt.imshow(matplotlib.image.imread(image_file))

Now, test on the downloaded image:

In [ ]:
t = time.time()
pred = test(image_file)
print("executed in %.3fs" % (time.time()-t))

pred_color = colorEncode(pred, loadmat(os.path.join(project_name, 'data/color150.mat'))['colors'])
plt.figure(figsize=(10, 5))
plt.imshow(pred_color)

## Second model: ResNet101dilated

### Download our second pretrained model

In [ ]:
ENCODER_NAME_2 = 'resnet101dilated'
DECODER_NAME_2 = 'ppm_deepsup'
EPOCH_2 = 25 # need to check model zoo table!!
BASE_URL_2 = f'http://sceneparsing.csail.mit.edu/model/pytorch/ade20k-{ENCODER_NAME_2}-{DECODER_NAME_2}'
PRETRAINED_ENCODER_MODEL_URL_2 = f'{BASE_URL_2}/encoder_epoch_{EPOCH_2}.pth'
PRETRAINED_DECODER_MODEL_URL_2 = f'{BASE_URL_2}/decoder_epoch_{EPOCH_2}.pth'

pretrained_encoder_file_2 = os.path.join(WEIGHTS_DIR, f'{ENCODER_NAME_2}-{DECODER_NAME_2}' + basename(PRETRAINED_ENCODER_MODEL_URL_2))
if not exists(pretrained_encoder_file_2) or os.path.getsize(pretrained_encoder_file_2) == 0:
  !wget -q -O "$pretrained_encoder_file_2" $PRETRAINED_ENCODER_MODEL_URL_2
pretrained_decoder_file_2 = os.path.join(WEIGHTS_DIR, f'{ENCODER_NAME_2}-{DECODER_NAME_2}' + basename(PRETRAINED_DECODER_MODEL_URL_2))
if not exists(pretrained_decoder_file):
  !wget -q -O "$pretrained_decoder_file_2" $PRETRAINED_DECODER_MODEL_URL_2

### Prepare ResNet101dilated

In [ ]:
net_encoder_2 = builder.build_encoder(arch=ENCODER_NAME_2, weights=pretrained_encoder_file_2,
                                      fc_dim=options.fc_dim)
net_decoder_2 = builder.build_decoder(arch=DECODER_NAME_2, weights=pretrained_decoder_file_2,
                                      fc_dim=options.fc_dim, num_class=options.num_class, use_softmax=True)
segmentation_module_2 = SegmentationModule(net_encoder_2, net_decoder_2,
                                           torch.nn.NLLLoss(ignore_index=-1)).eval()
if torch.cuda.is_available():
  segmentation_module_2 = segmentation_module_2.cuda()

### Evaluate ResNet101dilated

In [ ]:
t = time.time()
pred_2 = test(image_file, model=segmentation_module_2)
print("executed in %.3fs" % (time.time() - t))

colors = loadmat(os.path.join(project_name, 'data/color150.mat'))['colors']
plt.figure(figsize=(10, 5))
plt.imshow(colorEncode(pred_2, colors))

In [ ]:
options_scales = SimpleNamespace(**{**vars(options), 'imgSizes': [300, 375, 450, 525, 600]})

t = time.time()
pred_3 = test(image_file, opts=options_scales)  # original UperNet101, new scales
print("executed in %.3fs" % (time.time() - t))

plt.figure(figsize=(10, 5))
plt.imshow(colorEncode(pred_3, colors))

## Compare models

In [ ]:
runs = {
    'UperNet101 (original)':           (segmentation_module,   options),
    'ResNet101dilated + PPM':          (segmentation_module_2, options),
    'UperNet101, new test scales':     (segmentation_module,   options_scales),
}

fig, axes = plt.subplots(1, len(runs) + 1, figsize=(6 * (len(runs) + 1), 5))
axes[0].imshow(matplotlib.image.imread(image_file)); axes[0].set_title('Input')

for ax, (name, (model, opts)) in zip(axes[1:], runs.items()):
    if torch.cuda.is_available(): torch.cuda.synchronize()
    t = time.time()
    pred_cmp = test(image_file, model=model, opts=opts)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    ax.imshow(colorEncode(pred_cmp, colors))
    ax.set_title(f'{name}\n{time.time() - t:.2f}s')

for ax in axes: ax.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, 'comparison.png')); plt.show()